In [29]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px


In [30]:
img = cv2.imread('image.png')
mask = cv2.imread('mask.png', 0)



# 2. On crée un "stack" (un paquet) des deux images
# Pour que ça marche, le masque doit avoir 3 canaux comme l'image
mask_3ch = cv2.merge([mask, mask, mask])
combined = np.stack([cv2.cvtColor(img, cv2.COLOR_BGR2RGB), mask_3ch])

# 3. Affichage avec Plotly Express
# facet_col=0 dit à Plotly de séparer le "stack" en deux colonnes
fig = px.imshow(combined, facet_col=0, binary_string=True)

# 4. On nomme les axes et les titres
fig.layout.annotations[0].text = "Image Originale"
fig.layout.annotations[1].text = "Masque"

fig.update_layout(showlegend=False)
fig.show()



In [31]:
# pix_mask =[]
# for i in range(len(mask)) :
#     for j in range(len(mask[0])):
#         if mask[i,j] < 100 :
#             pix_mask.append([i,j])
# print(pix_mask)

mask = cv2.resize(mask, (img.shape[1], img.shape[0]))  # d'abord

pix_mask = []
for i in range(mask.shape[0]):
    for j in range(mask.shape[1]):
        if mask[i,j] < 100:
            pix_mask.append([i,j])

mask = cv2.resize(mask, (img.shape[1], img.shape[0]))
# mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)


# 3. Récupération du masque binaire (on prend l'index [1])
_, mask_only = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

# 4. Application du masque
# On fait un "ET" entre l'image et elle-même, mais UNIQUEMENT là où le masque est blanc
img_m = cv2.bitwise_and(img, img, mask=mask_only)

# 5. Affichage Plotly Express
# Conversion BGR -> RGB pour que les couleurs soient les bonnes
img_m_rgb = cv2.cvtColor(img_m, cv2.COLOR_BGR2RGB)
fig = px.imshow(img_m_rgb)
fig.update_layout(title="Aperçu de la zone masquée (Coordonnées actives)")
fig.show()




In [32]:
#algo

#noyeau 3 3
k = np.array([[1, 1, 1],
              [1, 0, 1],
              [1, 1, 1]])/8


k_5x5 = np.array([
    [1, 2,  4, 2, 1],
    [2, 4,  8, 4, 2],
    [4, 8,  0, 8, 4],  
    [2, 4,  8, 4, 2],
    [1, 2,  4, 2, 1]
])  
k_5x5 = k_5x5 / k_5x5.sum()  # normalisation propre



# def algo_oliveira(img, mask_indices):
#     image = img.astype(np.float32)
#     # On crée un masque binaire (1 pour image saine, 0 pour trou)
#     mask_map = np.ones(image.shape[:2], dtype=np.float32)
#     for y, x in mask_indices:
#         mask_map[y, x] = 0

#     for i in mask_indices:
#         y, x = i[0], i[1]
        
#         if 2 <= y < image.shape[0] - 2 and 2 <= x < image.shape[1] - 2:
#             # 1. On extrait les voisins de l'image ET du masque binaire
#             voisins = image[y-2:y+3, x-2:x+3]
#             voisins_valides = mask_map[y-2:y+3, x-2:x+3] # 1 si sain, 0 si masque
            
#             # 2. On ajuste le noyau : on ne garde que les poids des pixels SAINS
#             # On multiplie le noyau par le masque de validité
#             noyau_adapte = k_5x5 * voisins_valides
            
#             # 3. Normalisation CRUCIALE : 
#             # La somme des poids doit toujours être 1, mais seulement sur les pixels valides
#             somme_poids = np.sum(noyau_adapte)
            
#             if somme_poids > 0:
#                 noyau_normalise = noyau_adapte / somme_poids
#                 for c in range(3):
#                     image[y, x, c] = np.sum(voisins[:, :, c] * noyau_normalise)
    
#     return image.astype(np.uint8)






mask_bool = np.zeros(img_m_rgb.shape[:2], dtype=bool)
for y, x in pix_mask:
    mask_bool[y, x] = True

k_5x5 = np.array([
    [1, 2,  4, 2, 1],
    [2, 4,  8, 4, 2],
    [4, 8,  0, 8, 4], 
    [2, 4,  8, 4, 2],
    [1, 2,  4, 2, 1]
], dtype=np.float32) / 100

def algo_oliveira_opti(img, mask_indices, nb_iter=10):
    image = img.astype(np.float32)
    
    #pix a reparés
    current_mask = set(map(tuple, mask_indices))
    
    for iteration in range(nb_iter):
        nouveaux_pixels = {}
        
        for (y, x) in current_mask:
            if not (2 <= y < image.shape[0]-2 and 2 <= x < image.shape[1]-2):
                continue
            
            voisins = image[y-2:y+3, x-2:x+3]
            
            # Masque de validité : sain = pas dans current_mask
            voisins_valides = np.array([
                [0 if (y+dy-2, x+dx-2) in current_mask else 1
                 for dx in range(5)]
                for dy in range(5)
            ], dtype=np.float32)
            
            noyau_adapte = k_5x5 * voisins_valides
            somme_poids = np.sum(noyau_adapte)
            
            if somme_poids > 0:
                noyau_normalise = noyau_adapte / somme_poids
                pixel_value = np.array([
                    np.sum(voisins[:,:,c] * noyau_normalise)
                    for c in range(3)
                ]) #numpy array r g b >< liste, pour pouvoir faire operations vectorielles
                #seuil :si assez de voisins sains, on valide ce pixel
                ratio_sains = somme_poids  # entre 0 et 1
                nouveaux_pixels[(y, x)] = (pixel_value, ratio_sains)
        
        if not nouveaux_pixels:
            break
        
        #ajout nouveaux pix a image et supp dans mask
        for (y, x), (pixel_value, ratio) in nouveaux_pixels.items():
            image[y, x] = pixel_value
            if ratio > 0.3:  #seuil 
                current_mask.discard((y, x))
        
        print(f"It {iteration+1} : {len(current_mask)} pixels restants")
    
    return image.astype(np.uint8)

In [33]:
resultat = algo_oliveira_opti(img_m_rgb, pix_mask, nb_iter=100)
px.imshow(img_m_rgb).update_layout(title="Image depart").show()
px.imshow(resultat).update_layout(title="Image réparée").show()

It 1 : 2565 pixels restants
It 2 : 2414 pixels restants
It 3 : 2264 pixels restants
It 4 : 2123 pixels restants
It 5 : 1989 pixels restants
It 6 : 1857 pixels restants
It 7 : 1727 pixels restants
It 8 : 1596 pixels restants
It 9 : 1464 pixels restants
It 10 : 1334 pixels restants
It 11 : 1208 pixels restants
It 12 : 1084 pixels restants
It 13 : 966 pixels restants
It 14 : 849 pixels restants
It 15 : 733 pixels restants
It 16 : 618 pixels restants
It 17 : 507 pixels restants
It 18 : 405 pixels restants
It 19 : 318 pixels restants
It 20 : 223 pixels restants
It 21 : 122 pixels restants
It 22 : 64 pixels restants
It 23 : 21 pixels restants


In [ ]:
#algo oliviera classic 2001

def algo_oliveira(img, mask_indices, nb_iter=10):
    image = img.astype(np.float32)
    
    #pix a reparés
    mask = set(map(tuple, mask_indices))
    
    for iteration in range(nb_iter):
        nouveaux_pixels = {}
        
        for (y, x) in mask:
            if not (2 <= y < image.shape[0]-2 and 2 <= x < image.shape[1]-2):
                continue
            
            voisins = image[y-2:y+3, x-2:x+3]
    
            
            
            
            # Masque de validité pour garder que pix avec couleurs (empeche persistance du noir)
            masque_valeur = np.any(voisins > 0, axis=2).astype(np.float32)
            noyau_adapte = k_5x5 * masque_valeur
            somme_poids = np.sum(noyau_adapte)
            
            somme_poids = np.sum(noyau_adapte)
            
            if somme_poids > 0:
                noyau_normalise = noyau_adapte / somme_poids
                pixel_value = np.array([
                    np.sum(voisins[:,:,c] * noyau_normalise)
                    for c in range(3)
                ]) #numpy array r g b >< liste, pour pouvoir faire operations vectorielles
                
                
                nouveaux_pixels[(y, x)] = (pixel_value)
            
        
        
        
        for (y, x), pixel_value in nouveaux_pixels.items():
            image[y, x] = pixel_value
        
        print(f"It {iteration+1}")
    
    return image.astype(np.uint8)

In [37]:
resultat2 = algo_oliveira(img_m_rgb, pix_mask, nb_iter=100)
px.imshow(img_m_rgb).update_layout(title="Image depart").show()
px.imshow(resultat2).update_layout(title="Image réparée").show()

It 1
It 2
It 3
It 4
It 5
It 6
It 7
It 8
It 9
It 10
It 11
It 12
It 13
It 14
It 15
It 16
It 17
It 18
It 19
It 20
It 21
It 22
It 23
It 24
It 25
It 26
It 27
It 28
It 29
It 30
It 31
It 32
It 33
It 34
It 35
It 36
It 37
It 38
It 39
It 40
It 41
It 42
It 43
It 44
It 45
It 46
It 47
It 48
It 49
It 50
It 51
It 52
It 53
It 54
It 55
It 56
It 57
It 58
It 59
It 60
It 61
It 62
It 63
It 64
It 65
It 66
It 67
It 68
It 69
It 70
It 71
It 72
It 73
It 74
It 75
It 76
It 77
It 78
It 79
It 80
It 81
It 82
It 83
It 84
It 85
It 86
It 87
It 88
It 89
It 90
It 91
It 92
It 93
It 94
It 95
It 96
It 97
It 98
It 99
It 100
